# S&P 500 Sector Analysis — Intraday 5-min Data (2026-02-27)

This notebook loads the 5-minute intraday data for all S&P 500 stocks from last Friday,
groups them by GICS sector, and visualizes sector-level patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

## 1. Load Data

In [ ]:
constituents = pd.read_csv('../data/reference/sp500_constituents.csv')
intraday = pd.read_csv('../data/market/sp500_5min_2026-02-27.csv', index_col=0, parse_dates=True)

sector_map = constituents.set_index('Symbol')['GICS Sector'].to_dict()
intraday['Sector'] = intraday['Ticker'].map(sector_map)

print(f'Intraday rows: {len(intraday):,}')
print(f'Tickers: {intraday["Ticker"].nunique()}')
print(f'Sectors: {intraday["Sector"].nunique()}')
print(f'Time range: {intraday.index.min()} to {intraday.index.max()}')
intraday.head()

## 2. Sector Overview — Ticker Counts

In [ ]:
sector_counts = intraday.groupby('Sector')['Ticker'].nunique().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.tab20(np.linspace(0, 1, len(sector_counts)))
sector_counts.plot.barh(ax=ax, color=colors)
ax.set_xlabel('Number of Tickers')
ax.set_title('S&P 500 Tickers per GICS Sector')
for i, v in enumerate(sector_counts):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../output/sector/sector_ticker_counts.png', dpi=150)
plt.show()

## 3. Compute Per-Ticker Normalized Returns

For each ticker, normalize the Close price to its opening bar (first 5-min bar = 1.0),
so we can compare percentage moves across stocks and sectors.

In [ ]:
norm_parts = []
for ticker, group in intraday.groupby('Ticker'):
    group = group.sort_index()
    open_price = group['Close'].iloc[0]
    if open_price == 0 or pd.isna(open_price):
        group['NormReturn'] = np.nan
    else:
        group['NormReturn'] = (group['Close'] / open_price - 1) * 100
    norm_parts.append(group)

intraday = pd.concat(norm_parts)
print('NormReturn sample (AAPL):')
intraday[intraday['Ticker'] == 'AAPL'][['Ticker', 'Close', 'NormReturn']].head(10)

## 4. Sector Average Intraday Return Curves

Average the normalized returns across all tickers in each sector to see
how each sector moved throughout the day.

In [ ]:
sector_avg = intraday.groupby([intraday.index, 'Sector'])['NormReturn'].mean().unstack('Sector')

fig, ax = plt.subplots(figsize=(16, 8))
for col in sector_avg.columns:
    ax.plot(sector_avg.index, sector_avg[col], label=col, linewidth=1.3)

ax.axhline(0, color='black', linewidth=0.5, linestyle='-')
ax.set_title('S&P 500 Sector Avg Intraday Return (%) — 2026-02-27', fontsize=14)
ax.set_ylabel('Return from Open (%)')
ax.set_xlabel('Time (UTC)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../output/sector/sector_intraday_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Per-Sector Intraday Return Subplots

One subplot per sector showing every individual ticker as a thin line,
with the sector average in bold.

In [ ]:
sectors = sorted(intraday['Sector'].dropna().unique())
n_sectors = len(sectors)
ncols = 3
nrows = (n_sectors + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4), sharex=True)
axes_flat = axes.flatten()

for idx, sector in enumerate(sectors):
    ax = axes_flat[idx]
    sector_data = intraday[intraday['Sector'] == sector]

    for ticker, tdf in sector_data.groupby('Ticker'):
        ax.plot(tdf.index, tdf['NormReturn'], linewidth=0.3, alpha=0.35, color='steelblue')

    avg = sector_data.groupby(sector_data.index)['NormReturn'].mean()
    ax.plot(avg.index, avg.values, linewidth=2, color='darkred', label='Sector Avg')

    ax.axhline(0, color='black', linewidth=0.4)
    ax.set_title(sector, fontsize=11, fontweight='bold')
    ax.set_ylabel('Return (%)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.legend(fontsize=8)

for idx in range(n_sectors, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle('Individual Ticker Returns by Sector — 2026-02-27', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../output/sector/sector_individual_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sector Volume Profile

Total volume by sector over the course of the day.

In [ ]:
sector_volume = intraday.groupby([intraday.index, 'Sector'])['Volume'].sum().unstack('Sector').fillna(0)

fig, ax = plt.subplots(figsize=(16, 6))
x = sector_volume.index
bottom = np.zeros(len(x))
cmap = plt.cm.tab20(np.linspace(0, 1, len(sector_volume.columns)))

for i, col in enumerate(sector_volume.columns):
    ax.fill_between(x, bottom, bottom + sector_volume[col].values, alpha=0.7, label=col, color=cmap[i])
    bottom += sector_volume[col].values

ax.set_title('Sector Volume Profile — 2026-02-27', fontsize=14)
ax.set_ylabel('Total Volume')
ax.set_xlabel('Time (UTC)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../output/sector/sector_volume_profile.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. End-of-Day Sector Performance Summary

Final return (close vs open) for each sector, plus spread (best vs worst ticker).

In [ ]:
last_bar = intraday.groupby('Ticker').last()
last_bar['Sector'] = intraday.groupby('Ticker')['Sector'].first()

sector_summary = last_bar.groupby('Sector')['NormReturn'].agg(['mean', 'median', 'min', 'max', 'std']).round(2)
sector_summary.columns = ['Mean %', 'Median %', 'Min %', 'Max %', 'Std %']
sector_summary = sector_summary.sort_values('Mean %', ascending=False)

print(sector_summary.to_string())

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if v >= 0 else 'red' for v in sector_summary['Mean %']]
sector_summary['Mean %'].plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Mean Return from Open (%)')
ax.set_title('End-of-Day Sector Performance — 2026-02-27', fontsize=14)
for i, (val, std) in enumerate(zip(sector_summary['Mean %'], sector_summary['Std %'])):
    ax.text(val + (0.05 if val >= 0 else -0.05), i,
            f'{val:+.2f}% (\u00b1{std:.2f})', va='center', fontsize=9,
            ha='left' if val >= 0 else 'right')
plt.tight_layout()
plt.savefig('../output/sector/sector_eod_performance.png', dpi=150)
plt.show()

## 8. Sector Correlation Heatmap

How correlated were sector average returns throughout the day?

In [ ]:
corr = sector_avg.diff().dropna().corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=7)

fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Intraday Sector Return Correlation — 2026-02-27', fontsize=13)
plt.tight_layout()
plt.savefig('../output/sector/sector_correlation.png', dpi=150)
plt.show()